In [2]:
import torch
import torch.nn as nn
import torch.optim as optim
from torchvision import datasets, transforms
from torch.utils.data import DataLoader

In [3]:
# -----------------------
# Hyperparameters
# -----------------------
BATCH_SIZE = 128
EPOCHS = 40
LEARNING_RATE = 5e-4
WEIGHT_DECAY = 1e-4
DROPOUT_RATE = 0.1

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

In [4]:
# -----------------------
# Dataset (KMNIST)
# -----------------------
transform = transforms.Compose([
    transforms.ToTensor(),              # [0,1]
    transforms.Normalize((0.5,), (0.5,)) # normalize
])

train_dataset = datasets.KMNIST(
    root="./data",
    train=True,
    download=True,
    transform=transform
)

test_dataset = datasets.KMNIST(
    root="./data",
    train=False,
    download=True,
    transform=transform
)

train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True)
test_loader = DataLoader(test_dataset, batch_size=BATCH_SIZE)

In [5]:
# -----------------------
# MLP Model
# -----------------------
class MLP(nn.Module):
    def __init__(self):
        super().__init__()
        self.net = nn.Sequential(
            nn.Flatten(),
            nn.Linear(28 * 28, 128),
            nn.ReLU(),
            nn.Dropout(DROPOUT_RATE),

            nn.Linear(128, 64),
            nn.ReLU(),
            nn.Dropout(DROPOUT_RATE),

            nn.Linear(64, 10)
        )

    def forward(self, x):
        return self.net(x)

model = MLP().to(DEVICE)

# -----------------------
# Optimizer & Loss
# -----------------------
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(
    model.parameters(),
    lr=LEARNING_RATE,
    weight_decay=WEIGHT_DECAY
)


In [6]:
# -----------------------
# Training Loop
# -----------------------
for epoch in range(EPOCHS):
    model.train()
    total_loss = 0

    for images, labels in train_loader:
        images, labels = images.to(DEVICE), labels.to(DEVICE)

        optimizer.zero_grad()
        outputs = model(images)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()

        total_loss += loss.item()

    print(f"Epoch [{epoch+1}/{EPOCHS}] - Loss: {total_loss:.4f}")

# -----------------------
# Evaluation
# -----------------------
model.eval()
correct = 0
total = 0

with torch.no_grad():
    for images, labels in test_loader:
        images, labels = images.to(DEVICE), labels.to(DEVICE)
        outputs = model(images)

        probs = torch.softmax(outputs, dim=1)
        predictions = torch.argmax(probs, dim=1)

        correct += (predictions == labels).sum().item()
        total += labels.size(0)

accuracy = 100 * correct / total
print(f"\nTest Accuracy: {accuracy:.2f}%")

Epoch [1/40] - Loss: 352.9416
Epoch [2/40] - Loss: 185.3971
Epoch [3/40] - Loss: 144.8720
Epoch [4/40] - Loss: 119.6815
Epoch [5/40] - Loss: 104.0181
Epoch [6/40] - Loss: 92.4777
Epoch [7/40] - Loss: 83.5421
Epoch [8/40] - Loss: 78.0292
Epoch [9/40] - Loss: 71.6116
Epoch [10/40] - Loss: 66.0258
Epoch [11/40] - Loss: 61.6521
Epoch [12/40] - Loss: 56.8633
Epoch [13/40] - Loss: 54.1816
Epoch [14/40] - Loss: 51.8737
Epoch [15/40] - Loss: 50.3812
Epoch [16/40] - Loss: 46.1770
Epoch [17/40] - Loss: 44.1283
Epoch [18/40] - Loss: 41.5463
Epoch [19/40] - Loss: 40.4443
Epoch [20/40] - Loss: 40.4076
Epoch [21/40] - Loss: 38.4042
Epoch [22/40] - Loss: 36.8843
Epoch [23/40] - Loss: 36.5401
Epoch [24/40] - Loss: 36.0418
Epoch [25/40] - Loss: 33.1956
Epoch [26/40] - Loss: 32.3319
Epoch [27/40] - Loss: 32.5019
Epoch [28/40] - Loss: 30.9463
Epoch [29/40] - Loss: 31.1549
Epoch [30/40] - Loss: 28.4880
Epoch [31/40] - Loss: 28.3986
Epoch [32/40] - Loss: 28.7078
Epoch [33/40] - Loss: 27.6424
Epoch [34/40] 

In [17]:
idx = 1
image, label = test_dataset[idx]
image = image.unsqueeze(0).to(DEVICE)

model.eval()
with torch.no_grad():
    output = model(image)
    probs = torch.softmax(output, dim=1)
    confidence, pred = torch.max(probs, dim=1)

print(f"True Label     : {label}")
print(f"Prediction     : {pred.item()}")
print(f"Confidence     : {confidence.item() * 100:.4f}%")



True Label     : 9
Prediction     : 9
Confidence     : 100.0000%
